In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_1428_Okhla_Phase-2_Delhi_DPCC_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,195.29,314.34,11.86,57.68,40.44,71.59,7.40,1.29,17.38,...,NaN,11.90,72.55,0.53,204.15,0.00,0.00,32.87,992.84,NaN
1,2024-01-02,220.47,363.14,26.63,61.89,54.61,73.75,8.87,1.31,21.86,...,NaN,11.43,69.95,0.57,186.69,0.00,0.00,43.08,992.09,NaN
2,2024-01-03,220.86,344.80,28.52,63.70,57.12,90.53,7.63,1.93,15.51,...,NaN,10.94,80.82,0.57,179.29,0.00,0.00,28.65,991.94,NaN
3,2024-01-04,245.91,387.80,34.65,58.76,59.45,80.95,8.99,1.78,10.63,...,NaN,11.28,79.61,0.87,207.34,0.00,0.00,14.85,992.24,NaN
4,2024-01-05,179.52,303.73,22.55,60.33,50.43,58.35,7.74,1.20,16.02,...,NaN,11.81,82.89,0.87,198.18,0.00,0.00,10.75,992.09,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,201.58,272.01,72.65,51.05,86.18,56.23,6.40,1.64,17.03,...,NaN,16.80,84.36,0.38,222.52,0.26,0.26,6.16,991.00,NaN
362,2024-12-28,93.52,134.61,46.07,46.86,62.39,54.74,9.47,1.88,8.99,...,NaN,16.88,87.71,0.33,190.21,0.02,0.02,9.48,991.00,NaN
363,2024-12-29,106.08,157.88,16.39,29.74,29.10,50.22,6.76,0.83,20.87,...,NaN,16.65,81.23,0.66,251.69,0.00,0.00,47.41,991.00,NaN
364,2024-12-30,106.00,162.33,19.56,33.51,33.71,42.18,5.46,1.57,24.70,...,NaN,15.05,78.09,0.47,239.10,0.00,0.00,44.48,991.00,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
RF (mm)            0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         195.29        314.34       11.86        57.68   
1  2024-01-02         220.47        363.14       26.63        61.89   
2  2024-01-03         220.86        344.80       28.52        63.70   
3  2024-01-04         245.91        387.80       34.65        58.76   
4  2024-01-05         179.52        303.73       22.55        60.33   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      40.44       71.590         7.40        1.29          17.38   
1      54.61       73.750         8.87        1.31          21.86   
2      57.12       44.745         7.63        1.93          15.51   
3      59.45       80.950         8.99        1.78          10.63   
4      50.43       58.350         7.74        1.20          16.02   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0             3.07            12.98    11.90   72.55      0

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,1.644268,0.849372,-0.729918,-0.028335,-0.550036,1.834486,-0.380146,-0.575411,-0.965931,0.715449,-0.338692,-1.791472,0.909071,-0.662496,0.523533,0.0,0.0,-1.151923,1.295544
1,2024-01-02,2.023341,1.275749,-0.150189,0.162794,-0.112062,1.988290,0.072636,-0.520738,-0.677294,0.839226,-0.244167,-1.852027,0.726529,-0.562082,0.122981,0.0,0.0,-0.899170,1.187254
2,2024-01-03,2.029213,1.115508,-0.076005,0.244966,-0.034482,-0.077021,-0.309303,1.174126,-1.086412,1.407145,-0.180861,-1.915159,1.489695,-0.562082,-0.046783,0.0,0.0,-1.256391,1.165596
3,2024-01-04,2.406329,1.491209,0.164600,0.020696,0.037535,2.500968,0.109598,0.764078,-1.400820,0.802821,-0.124493,-1.871353,1.404743,0.191019,0.596715,0.0,0.0,-1.598016,1.208912
4,2024-01-05,1.406858,0.756670,-0.310331,0.091972,-0.241260,0.891727,-0.275421,-0.821440,-1.053553,0.336836,-0.146173,-1.803068,1.635027,0.191019,0.386575,0.0,0.0,-1.699513,1.187254
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,1.738961,0.479526,1.656118,-0.329328,0.863720,0.740772,-0.688161,0.381367,-0.988481,1.225120,1.073978,-1.160155,1.738233,-1.039046,0.944961,0.0,0.0,-1.813140,1.029872
362,2024-12-28,0.112169,-0.720968,0.612840,-0.519549,0.128406,0.634676,0.257445,1.037443,-1.506482,0.132968,0.233661,-1.149847,1.973432,-1.164563,0.203734,0.0,0.0,-1.730952,1.029872
363,2024-12-29,0.301254,-0.517653,-0.552114,-1.296775,-0.900539,0.312828,-0.577276,-1.832891,-0.741077,-0.427670,-0.715924,-1.179481,1.518481,-0.336152,1.614154,0.0,0.0,-0.791979,1.029872
364,2024-12-30,0.300049,-0.478772,-0.427690,-1.125622,-0.758051,-0.259663,-0.977695,0.190011,-0.494318,-0.449513,-0.710721,-1.385625,1.298026,-0.813116,1.325325,0.0,0.0,-0.864513,1.029872


In [10]:
df.to_excel("okhla2024.xlsx", index=False)